In [1]:
'''New mechanism of identification based on the table of example analysis'''
'''create polynomial from all of the examples'''
'''try to do note_simplify with numbers if note sequential'''
'''add corrections for notsequential functions. Fix model'''

'''found mistake for symmetrical structures. I save as F one full column, but 
it is needed to save F1 and F2 and only unique of them. Need to change 
mechanism of parametrs storing and filling data_step
made correction. Save results and dictm on each step for F construction

Need to clean code and add iterative mechanism branch moving
'''
# 220128 исправил формирование таблицы конфликтов
# 220203 нужно проверить все ли решения в пуле максимальны. Думаю, что эт не так и нужно будет их доп. проверять
# 220208 переписываю на использование стандартной note_simplify_id. 
# Нужно настроить решатель на работу сполиномами большой степени. Пробуй через логические операции
# 220213 ркшает, похоже, но треккер говорит, что есть ошибки
# 220214 изменения в порядок расположения элементов в матрицах. Теперь результат как завел НА
# note_simplify_id changed
# 220218 веротно нашел  ошибку - свернутая матрица заполняется не по порядку
# у меня где-то был код заполнения матрицы значениями по именам. Используй.

import re
#import functions_is as fis

import numpy as np
import pandas as pd
import string
from itertools import chain
#model.Params.Threads = 8
import gurobipy as gp
from gurobipy import GRB
import sys
import math
import copy

In [2]:
#BinToDec() == int("000110", 2) # конвертирует bin in int
#’a = {0:10b}’.format(a) #bina

def listToString(list1):
    return (''.join(map(str, list1)))

def listToString_s(list1):
    return (' '.join(map(str, list1)))

def listToString_plus(list1):
    return ('+'.join(map(str, list1)))



def Bin2Pos(x):#DectoMyNum(x):
    '''принимаю one-hot значение(бинарный формат) в виде строки
    возвращаю позицию единицы
    "00001" == 0'''
    x = int(x,2)
    for i in range(10):
        if x == pow(2, i): return(i)
    return(0)

def oh2i(vec): 
    '''convert 1h (integer vector, list) to integer number. Simply return position of 1 in vector  '''
    vec = list(map(round, vec)) # for float input convert into int
    for i in range(len(vec)): 
        if vec[i] == 1: return(i)
    return(0) 

def ohvec2i(vec, dim):
    '''conver oh list into int list
    parts of i shoul be together in vec'''
    res = []
    i = 0
    while i < len(vec):
        t = []
        for j in range(dim):
            t.append(vec[i+j])
        t = oh2i(t)
        res.append(t)
        i += dim
    return res
#r = ohvec2i([1,0,0,0,0,1], 3)

# def ohvec2i_dict(vec, dim, dict_i):
#     '''conver oh list into int list
#     parts of i get from dict'''
#     res = []
#     i = 0
#     while i < len(vec):
#         t = []
#         for j in range(dim):
#             t.append(vec[i+j])
#         t = oh2i(t)
#         res.append(t)
#         i += dim
#     return res

def i2oh(i, dim):
    if i >= dim: i = dim - 1 
    elif i < 0: i = 0
    vec = [0 for a in range(dim)]
    vec[i] = 1
    return vec

def RowDimToRange(y,dim):
    '''get pow of 2 and dim - return range((dim*i+1),(dim*(i+1)), from 0'''
    for i in range(10):
        if y == pow(2, i): return( range((dim*i),(dim*(i+1))))
    return(0)

def RowDimToRange_2(y,dim):
    '''get integer and dim - return range((dim*(i-1)+1),(dim*i)), from 0'''
    for i in range(0, 10):
        if y == i: return( range((dim*i),(dim*(i+1))))
    return(0)

def selectFromDF(dframe, left, right, dim):
    '''get selection from np.array, according to dimension
    уномение y*dframe*x, y - м.б. 0, когда нужно умножение только с одним множетелем
    добавь вариант когда х - ноль
    '''
    if not(sum(left)): range_left = range(dframe.shape[0]) # y = zero, so get all rows
    else:
        #row <- DectoMyNum(y)
        leftd_r = oh2i(left) # analize were 1 position in left term
        #print("left", leftd_r)
        range_left = RowDimToRange_2(leftd_r, dim)
        #print("r_left", range_left)
  
    if not(sum(right)): range_right = range(dframe.shape[1]) # x = zero, so get all columns
    else:
        range_right = oh2i(right) # num of 1 in vec
  
    dframe = dframe[range_left,range_right]
    return(dframe.tolist())

def Fcomb(df_str, df_col, dim): 
    '''формируем столбец результата умножения строки на столбец для текстовых переменных
    считаю что размерность матрицы и ячеек матрицы - совпадают. row and col - linear
    на вход получаю 2 списка. Выдаю список'''
    Rdf = [i for i in range(dim)]
    txt = [i for i in range(dim)]
    for m in range(dim):    #перебор колонки результата
        for i in range(dim):  # перебор по строкам
            range_str = RowDimToRange_2(0,dim)
            range_col = RowDimToRange_2(i,dim) # column data is complex, so there is range and recounting in range
            txt[i] = str(df_str[range_str[i]]) + "*" + str(df_col[range_col[m]]) # диапазон определяется через i а позиция в диапазоне через m
   
        Rdf[m] = "(" +  listToString_plus(txt) + ")"
    return(Rdf)

def m_create(numofletter, dim):
    '''create array with numbered simbols, from letter'''
    m_t = np.empty((dim*dim,dim), dtype = 'U16')
    for j in range(dim):
        for i in range(dim):
            for d in range(dim):
                m_t[i*dim+d,j] = '{}{}{}_{}'.format(string.ascii_lowercase[numofletter], j, i, d)
    return m_t 

def m_create_byname(name, dim):
    '''create array with numbered simbols, from letter'''
    m_t = np.empty((dim*dim,dim), dtype = 'U16')
    for j in range(dim):
        for i in range(dim):
            for d in range(dim):
                m_t[i*dim+d,j] = '{}{}{}_{}'.format(name, j, i, d)
    return m_t 

def m_create_byname_t(name, dim):
    '''create array with numbered simbols, from letter
    in NA style'''
    m_t = np.empty((dim*dim,dim), dtype = 'U16')
    for i in range(dim):
        for j in range(dim):
            for d in range(dim):
                m_t[i*dim+d,j] = '{}{}{}_{}'.format(name, j, i, d)
    return m_t 

def l_create(dim): return [i for i in range(dim)]

def l_create_0oh(dim):
    '''return one-hot zero vector '''
    zero = [1]
    zero = zero + [0 for i in range(dim-1)]
    return zero

def ll_create_0oh(l_num, dim):
    list_l = []
    for m in range(l_num):
        list_l.append(l_create_0oh(dim))
    return list_l
    
def lm_create(m_num, dim):
    '''return list of np matrises. Contain chars in one-hot style'''
    list_m = []
    for m in range(m_num):
        list_m.append(m_create(m, dim))
    return list_m

def list_split(ll, dim):
    if len(ll)%dim: print("incorrect list splitting")
    lol = [[] for i in range(int(len(ll)/dim))]
    cnt = 0
    for i in range(len(ll)):
        if i > 0 and not i%dim:
            cnt += 1
        lol[cnt].append(ll[i])
        
    return lol
# попробуй преобразовать а массив а затем изметить форму


In [3]:

def shell_find(f_text_v, mark=0):
    '''return coordinates of upper layer brakets in phrase'''
    Term1_r_brct_ind, Term1_l_brct_ind, Term1_r_brct_cnt, Term1_l_brct_cnt = 0, 0, 0, 0
    Term1_l_brct_found = 0
    while mark < len(f_text_v):         
        if f_text_v[mark] == '(': 
            Term1_l_brct_cnt = Term1_l_brct_cnt + 1
            if not Term1_l_brct_found:
                Term1_l_brct_ind = mark 
                Term1_l_brct_found = 1
        if f_text_v[mark] == ')':
            if Term1_r_brct_cnt >= Term1_l_brct_cnt: return -1, -1 # like ...)*()
            Term1_r_brct_cnt = Term1_r_brct_cnt + 1
            Term1_r_brct_ind = mark    
        if (Term1_l_brct_cnt != 0) and (Term1_r_brct_cnt == Term1_l_brct_cnt): break
        mark = mark + 1
    if Term1_l_brct_cnt != Term1_r_brct_cnt: return -1, -1
    else: return Term1_l_brct_ind, Term1_r_brct_ind

def rbracket_find(f_text_v, mark=0):
    '''return coordinates of upper layer right braket in phrase'''
    Term1_r_brct_ind, Term1_l_brct_ind, Term1_r_brct_cnt, Term1_l_brct_cnt = 0, 0, 0, 0
    Term1_l_brct_found = 0
    while mark < len(f_text_v):         
        if f_text_v[mark] == '(': 
            Term1_l_brct_cnt = Term1_l_brct_cnt + 1
            if not Term1_l_brct_found:
                Term1_l_brct_ind = mark 
                Term1_l_brct_found = 1
        if f_text_v[mark] == ')':
            if Term1_r_brct_cnt >= Term1_l_brct_cnt: return -1 # like ...)*()
            Term1_r_brct_cnt = Term1_r_brct_cnt + 1
            Term1_r_brct_ind = mark    
        if (Term1_l_brct_cnt != 0) and (Term1_r_brct_cnt == Term1_l_brct_cnt): break
        mark = mark + 1
    if Term1_l_brct_cnt != Term1_r_brct_cnt: return -1
    else: return Term1_r_brct_ind

    
def get_monomials(f_text):
    '''return list of monoms (was named substr_monomial)'''
    doc = re.sub("\+", ' ', f_text) 
    doc = re.sub("\)|\(", ' ', doc)
    vals = doc.split()
    vals  = list(dict.fromkeys(vals))
    vals.sort()
    return(vals)

def get_productparts(f_text):
    '''return list of monoms (was named substr_monomial)'''
    doc = re.sub("\*", ' ', f_text) 
    doc = re.sub("\)|\(", ' ', doc)
    vals = doc.split()
    vals  = list(dict.fromkeys(vals))
    vals.sort()
    return(vals)

def getitem(text):
    '''return first expression before +*()'''
    i = 0
    while i < len(text):
        if text[i] == '+' or text[i] == '*' or text[i] == '(' or text[i] == ')':
            return text[0:i]
        i += 1
    return text
#getitem('a_0012333*b00_3')

def polynomial_simplify(f_text_v):
    '''It opens all brackets in polynomial'''
    
    # USE get item function for item_lgth
    item_lgth = getitem(f_text_v[1:len(f_text_v)])
    item_lgth = len(item_lgth) 
    
    # like a00_0. Better to detect item_lgth
    Term1_r_brct_ind, Term1_l_brct_ind, Term1_r_brct_cnt, Term1_l_brct_cnt = [0 for _ in range(4)]
    Term2_r_brct_ind, Term2_l_brct_ind, Term2_r_brct_cnt, Term2_l_brct_cnt = [0 for _ in range(4)]
    err_code, mark = 0, 0
    store_1, store_2, str_sum, str_new = ["" for _ in range(4)]
    if(not re.search('\(', f_text_v)): return(f_text_v)
  
    while(mark < len(f_text_v)): 
        # better stop condition?
        Term1_l_brct_ind, Term1_r_brct_ind = shell_find(f_text_v, mark)
        mark = Term1_r_brct_ind
        if(max(Term1_l_brct_ind, Term1_r_brct_ind) == 0):
            #break  # there are no brackets in phrase
            if(str_new == ""): return(f_text_v) # first time on this recursion lvl
            else: return(str_new) # there was result on this lvl

        if(max(Term1_l_brct_ind, Term1_r_brct_ind) == -1):
            err_code <- 1
            break
    
        if( (Term1_r_brct_ind < len(f_text_v)-1) and (f_text_v[mark+1] == '*') ):
            # ()*smth: ()*a or ()*() 
            if(f_text_v[mark+2] != "("): 
                # monomial next after bracket
                Term2_l_brct_ind = mark+2
                # USE get item function for item_lgth
                item_lgth = getitem(f_text_v[mark+2:len(f_text_v)])
                item_lgth = len(item_lgth) 
                Term2_r_brct_ind = mark+2+item_lgth
                mark = Term2_r_brct_ind
            else: 
                # another () next after first bracket
                Term2_l_brct_ind = mark+3
                # +3 because I whant to get the brackets content
                Term2_r_brct_ind = rbracket_find(f_text_v, mark+2)
                mark = Term2_r_brct_ind
            # get what in brackets, without "(" ")"
            store_1 = polynomial_simplify(f_text_v[Term1_l_brct_ind+1:Term1_r_brct_ind]) 
            store_2 = polynomial_simplify(f_text_v[Term2_l_brct_ind:Term2_r_brct_ind])
            # get valute next after bracket # even if it is ()
            store_1 = get_monomials(store_1)
            store_2 = get_monomials(store_2)
            mult = [] 
            # monomial multiplication
            for i in range(len(store_1)):
                for j in range(len(store_2)): mult.append(store_1[i] + "*" + store_2[j])
            str_sum = "+".join(mult)
            # monomial summation
            if(str_new == ""): str_new = str_sum 
            else: str_new = "+".join([str_new, str_sum])  
            # remove used part
            f_text_v = f_text_v[0:Term1_l_brct_ind] + f_text_v[Term2_r_brct_ind+1:len(f_text_v)]
            mark = max(mark - Term2_r_brct_ind, 0)
    
        elif( (Term1_l_brct_ind > 0) and (f_text_v[Term1_l_brct_ind-1] == '*') ): 
            # a*() # этот случай не входит в первое условие видимо потому что алгорим ищет скобки, для вхождения в анализ
            # encoding: Term2*(Term1)
            # get item before '('
            ii = Term1_l_brct_ind-2
            while ii > 0:
                if f_text_v[ii] == '+' or f_text_v[ii] == '*' or f_text_v[ii] == '(' or f_text_v[ii] == ')':
                    break
                ii -= 1
            Term2_l_brct_ind = ii
            #Term2_l_brct_ind = Term1_l_brct_ind-1-item_lgth
            Term2_r_brct_ind = Term1_l_brct_ind-1
            # get what in brackets, without "(" ")"
            store_1 = polynomial_simplify(f_text_v[Term1_l_brct_ind+1:Term1_r_brct_ind]) 
            store_2 = polynomial_simplify(f_text_v[Term2_l_brct_ind:Term2_r_brct_ind])
            # get valute next after bracket # even if it ()
            store_1 = get_monomials(store_1)
            store_2 = get_monomials(store_2)
            mult = [] 
            # monomial multiplication
            for i in range(len(store_1)):
                for j in range(len(store_2)): mult.append(store_1[i] + "*" + store_2[j])
            str_sum = "+".join(mult) 
            # monomial summation
            if(str_new == ""): str_new = str_sum 
            else: str_new = "+".join([str_new, str_sum])
            f_text_v = f_text_v[0:Term2_l_brct_ind] + f_text_v[Term1_r_brct_ind+1:len(f_text_v)]
            # remove used part
            mark = max(mark - Term1_r_brct_ind, 0)
        else: 
            # +()+ case
            store_1 = polynomial_simplify(f_text_v[Term1_l_brct_ind+1:Term1_r_brct_ind]) 
            if(str_new == ""): str_new = store_1 
            else: str_new = "+".join([str_new, store_1])
      
            f_text_v = f_text_v[0:Term1_l_brct_ind] + f_text_v[Term1_r_brct_ind+1:len(f_text_v)]
            mark = max(mark - Term1_r_brct_ind, 0)

    return(str_new)



In [4]:
def note_simplify_id(note, list_l, list_M, dim):
    '''Core function for character multiplication 
    in IRM identification'''
    i = len(note)-1 # № of M in note
    res = 0
  
    while note[i][0] != "M": i = i - 1
    numOfList_1 = int(note[i+1][1:len(note[i+1])]) - 1
    numOfList_2 = int(note[i+2][1:len(note[i+2])]) - 1
    # - 1 needs because lists in note enumeted from 1
    M = list_M[len(list_M)-1]
    #print('M', M)
    list_1 = list_l[numOfList_1]
    if i + 2 <= len(note):
        # there are 2 lists or no Matr
        list_2 = list_l[numOfList_2]
        # get info about vectors in list positions
        list_1_type = isinstance(list_1[1], str)
        list_2_type = isinstance(list_2[1], str)
        # caculate result in different situations
        if not(list_1_type) and not(list_2_type):
            #print('both num   ')
            # both numerical # "types: FALSE FALSE"
            res = selectFromDF(M, list_2, list_1, dim) 
    
        if (list_1_type == False) and (list_2_type == True): 
            # 1 - numerical, 2 - str
            res = selectFromDF(M, [0], list_1, dim) 
            res = Fcomb(list_2, res, dim) 
    
        if list_1_type and list_2_type: 
            #print('both str   ')
            # both str      
            # get M by strings(oh, wide = dim) and throgth Fcomb form one column
            res_comb_t = []
            res = []
            for k in range(dim):
                v = []
                for ii in range(dim):
                    for j in RowDimToRange_2(k,dim):
                        #v = np.hstack(v, M[j,ii])
                        v.append(M[j,ii])
                #print(v)
                res = Fcomb(list_1, v, dim)
                #print('res_l1', res)
                #res_comb_t = np.hstack(res_comb_t, res)
                #res_comb_t.append(res)
                res_comb_t = list(chain(res_comb_t, res))
                # w with lists so have row in output 
            #print("res_comb_t", res_comb_t)
            res = Fcomb(list_2, res_comb_t, dim) 
            #print('res_l2', res)
            # M1*res_2 #vec(2)*vec(4) -> vec(2)

    note[i] = "l{}".format(numOfList_1+1)
    #if i + 3 <= len(note): note = note[0:i+1] + note[i+2:len(note)] # don't see why
    note = note[0:i+1] + note[i+2:len(note)] # del i+1 item
    note = note[0:i+1] + note[i+2:len(note)] # del i+2 item
    return note, res, numOfList_1, numOfList_2

def note_simplify_id_onestep(note, list_1, list_2, M, dim):
    '''Core function for character multiplication 
    in IRM identification. 2 lists mult on M'''
    i = len(note)-1 # № of M in note
    res = 0

    # get info about vectors in list positions
    list_1_type = isinstance(list_1[1], str)
    list_2_type = isinstance(list_2[1], str)
    # caculate result in different situations
    if not(list_1_type) and not(list_2_type): 
        # both numerical # "types: FALSE FALSE"
        res = selectFromDF(M, list_2, list_1, dim) 
    if (list_1_type == False) and (list_2_type == True): 
        res = selectFromDF(M, [0], list_1, dim) 
        res = Fcomb(list_2, res, dim) 
    if list_1_type and list_2_type: 
        # both str      
        # get M by strings(oh, wide = dim) and throgth Fcomb form one column
        res_comb_t = []
        res = []
        for k in range(dim):
            v = []
            for ii in range(dim):
                for j in RowDimToRange_2(k,dim):
                    #v = np.hstack(v, M[j,ii])
                    v.append(M[j,ii])
            res = Fcomb(list_1, v, dim)
            res_comb_t = list(chain(res_comb_t, res))
        res = Fcomb(list_2, res_comb_t, dim) 
        # M1*res_2 #vec(2)*vec(4) -> vec(2)

    return res

In [5]:
def makedimension(name, dim):
    '''create list w dimensions of name'''
    lname = []
    for i in range(dim):
        lname.append(name+'_'+str(i))
    return lname

def select_monem(text, i):
    if text[i].isalpha(): i += 1
    else: return i
    while i < len(text):
        if text[i].isnumeric():
            i += 1
            continue
        else: break
    return i

def traverse_note_get_branches(container, which_branch):  
    '''return 1 or 2 subtree. #211119 changed
        1 - for the right, 2 -for left
        container(list) = text, i, branch'''
    branch_r = ""
    branch_l = ""
    Matr = ""
    #print(container[0][container[1]])
    
    if container[0][container[1]].isalpha():
        edge = select_monem(container[0], container[1])
        monem = container[0][container[1]:edge]
    else: return container[0][container[1]]
    #print(monem)
    if container[0][container[1]] == 'l':
        container[2] = monem
        #container[2] = digit2alpha(monem[1:len(monem)])
        if(edge < len(container[0])): container[1] = edge
        else: container[1] = len(container[0]) - 1 
        return(container)
    else:
        if(which_branch == 0): Matr = monem
        #if(which_branch == 0): Matr = '('
        if(edge < len(container[0])): container[1] = edge
        else: container[1] = len(container[0]) - 1 
  
    if(which_branch == 0): # std mode
        container = traverse_note_get_branches(container, 0) 
        branch_r = container[2]
        #print("branch_r", branch_r)
#         if branch_r[0] == 'M':
#             container[3].append(branch_r)
        
        container = traverse_note_get_branches(container, 0)  # left
        branch_l = container[2]
        #print("branch_l", branch_l)
#         if branch_l[0] == 'M':
#             container[3].append(branch_l)
        container[3].append([Matr, branch_r, branch_l])
    else:
        container = traverse_note_get_branches(container, 0) 
        branch_r = container[2]
    
        container = traverse_note_get_branches(container, 0)  # left
        branch_l = container[2]
        if(which_branch == 1):branch_l = ""
        if(which_branch == 2):branch_r = ""
        which_branch = 0 # do once, for now
  
  
    container[2] = Matr + branch_r + branch_l
    #print(container[[3]])
    return(container)


def traverse_note_get_branches_wm(container, which_branch, m_upper):  
    '''return 1 or 2 subtree. #212104 changed
        1 - for the right, 2 -for left
        container(list) = text, i, branch, output
        output = core node, matr w her branches: l r'''
    branch_r = ""
    branch_l = ""
    Matr = ""
    #print(container[0][container[1]])
    
    if container[0][container[1]].isalpha():
        edge = select_monem(container[0], container[1])
        monem = container[0][container[1]:edge]
    else: return container[0][container[1]]
    #print(monem)
    if container[0][container[1]] == 'l':
        container[2] = monem
        #container[2] = digit2alpha(monem[1:len(monem)])
        if(edge < len(container[0])): container[1] = edge
        else: container[1] = len(container[0]) - 1 
        return(container)
    else:
        if(which_branch == 0): Matr = monem
        #if(which_branch == 0): Matr = '('
        if(edge < len(container[0])): container[1] = edge
        else: container[1] = len(container[0]) - 1 
  
    if(which_branch == 0): # std mode
        container = traverse_note_get_branches_wm(container, 0, Matr) 
        branch_r = container[2]
        
        container = traverse_note_get_branches_wm(container, 0, Matr)  # left
        branch_l = container[2]

        container[3].append([m_upper, Matr, branch_r, branch_l])
  
    container[2] = Matr + branch_r + branch_l
    #print(container[[3]])
    return(container)

def get_leaves_numbers_txt(branch):
    '''return leaves numbers from lm word (grpfrombranch)
    return list of str'''
    lon = []
    i = 0
    while i < len(branch):
        if branch[i] == 'l': 
            i += 1
            num = ''
            while branch[i].isdigit(): 
                num += branch[i]
                if i < len(branch)-1: i += 1
                else: break
            lon.append(num)
        else:
            i += 1                
    return lon

def get_leaf_number(branch):
    '''return leaf numbers from lm word (grpfrombranch)
    return int'''
    res = 0
    i = 0
    while i < len(branch):
        if branch[i] == 'l': 
            i += 1
            num = ''
            while branch[i].isdigit(): 
                num += branch[i]
                if i < len(branch)-1: i += 1
                else: break
            res = int(num)
        else:
            i += 1              
    return res

def get_leaves_txt(branch):
    '''return leaves numbers from lm word (grpfrombranch)
    return list of str with l letters'''
    lon = []
    i = 0
    while i < len(branch):
        if branch[i] == 'l': 
            i += 1
            num = 'l'
            while branch[i].isdigit(): 
                num += branch[i]
                if i < len(branch)-1: i += 1
                else: break
            lon.append(num)
        else:
            i += 1
                
    return lon

def get_matrix_txt(branch):
    '''    return M + num'''
    num = ''
    i = 0
    if branch[i] == 'M': 
        num = 'm'
        i += 1
        while branch[i].isdigit(): 
            num += branch[i]
            if i < len(branch)-1: i += 1
            else: break
    else:
        num = 'u'
        i += 1
                
    return num

def makerestriction_pair(lstore): # 211216
    '''input - lists pair. output - product'''
    lres = []
    # умножение каждый с каждым
    for i in range(len(lstore)):
        for j in range(i+1,len(lstore)):
            #if i == j: continue
            lres.append([n+'*'+m for n in lstore[i] for m in lstore[j]])
    return list(chain.from_iterable(lres))


In [8]:
#data_ini = pd.read_csv('211208_football_data8.csv', sep=';')
#data_ini = pd.read_csv('211209_id_arctic_l5_data.csv', sep=';')
data_ini = pd.read_csv('freeley_tab_d4_cl3_h.csv', sep=';')
print(data_ini)
lnum = data_ini.shape[1]-1
print("lnum = ", lnum)
data = np.array(data_ini.iloc[:,0:lnum+1])
#print("data: \n" ,data)
dim = 1+data.max()-data.min()
print("dim = ", (data.max()-data.min())+1)
exnum = len(data_ini)

#note = 'M1l1M2l2M3l3l4' # 1 no solution
#note = 'M1l1M2l4M3l2l3' # 3 solution w the same matrices as previous
#note = 'M1l2M2l1M3l3l4' # 4
#note = 'M1l1M2l4M3l2l3'
#note = 'M1M2l1l2M3l3l4' # 13 solution w the diffenent matrices
#note = 'M1M2l1l3M3l2l4' # 14 no solution
#note = 'M1M2l1l4M3l2l3' # 15 no solution
#note = 'M1l1M2l2M3l3M4l4l5' # 1
#note = 'M1l2M2l1M3l3M4l4l5' # 13
#note = 'M1l5M2M3l1l4M4l2l3' # 75
#note = 'M1M2l1l2M3l3M4l4l5' # 76
#note = 'M1l1M2l2M3l3M4l4M5l5M6l6M7l7M8l8M9l9l10'
#note = 'M1M2l1l2M3l3M4l4M5l5M6l6M7l7M8l8M9l9l10'
#note = 'M1M2M3l1l2M4l3l4M5l5M6l6M7l7M8l8M9l9l10'

#lnum = len(get_leaves_txt(note))
#solvesteps = traverse_note_get_branches_wm([note, 0, "", []], 0, 'F')[3]
#print('solvesteps:', solvesteps)

# отсортировать списки по номеру матрицы первого элемента: М1, М2 ....
solvesteps_s = []
# for i in range(lnum):
#     seek = 'M'+str(i)
#     if i == 0: seek = 'F'
#     for term in solvesteps:
#         if seek in term[0]:
#             solvesteps_s.append(term)
# print(solvesteps_s)  
#solvesteps_s = [['F','M1','l1','M2l2m3']]
#solvesteps_s = ['M1', 'l1', 'M2', 'l4', 'm5'] # рассматриваю только случаи когда сворачивается хвост записи
#solvesteps_s = ['M1', 'l3', 'M2', 'l4', 'm5'] #9
# м5 - обозначение свертки. м - значит сверткаб 5 - значит хранить в 5-й ячйке списка листов
#currmatrnum = 2
#appel_list = ['l1', 'l2']

#solvesteps_s = ['M1','l2','M2', 'l1', 'M3', 'l3', 'M4', 'l4', 'l5'] # 13 arct
solvesteps_s = ['M1','l1','M2','l2','M3','l3','M4','l4','M5','l5','M6','l6','M7','l7','M8','l8',
               'M9','l9','l10'] 
#solvesteps_s = ['M1','l1','M2', 'l2', 'm6'] # 1 arct
currmatrnum = 9
#appel_list = ['l3', 'l4', 'l5']
appel_list = []
len(appel_list)


    l1  l2  l3  l4  l5  l6  l7  l8  l9  l10  F
0    1   0   0   1   1   1   1   1   0    0  2
1    1   0   0   1   1   1   1   1   2    0  0
2    1   0   0   1   1   1   1   1   1    0  2
3    0   0   1   1   1   1   0   1   1    1  2
4    1   1   1   1   1   1   0   0   0    0  2
5    1   0   1   1   1   1   0   0   0    1  2
6    1   0   1   1   1   1   0   0   0    0  1
7    2   1   2   1   1   1   1   1   1    0  2
8    2   1   1   1   1   1   1   1   0    1  2
9    2   1   1   1   0   1   1   1   1    0  2
10   2   1   1   1   1   1   1   1   1    0  2
11   1   1   2   1   1   1   0   1   2    1  2
12   1   1   2   1   1   1   0   1   2    0  0
13   0   0   1   0   0   1   0   0   1    0  1
14   0   0   1   1   1   1   0   1   2    0  2
15   0   0   1   1   1   0   0   1   2    1  1
16   0   0   1   1   1   0   0   1   1    1  2
17   1   0   1   1   1   1   1   1   0    0  2
18   1   0   1   1   1   1   1   1   1    0  2
19   1   0   2   0   1   1   0   0   0    0  2
20   1   0   

0

In [9]:
# the main cycle
# solvesteps_s = [['M1', 'M2l1l2', 'M3l3l4']]
print("следи за правильностью заполнения хвоста")
data_step = data_ini
list_M = []
list_M_vals = [] # M results on each step in sequence form
list_M_vals_m = [] # M results on each step in matrix form
#list_mvar = [] # variable results on each step. On first it is the F. in int format
#list_mvar = [[] for i in range(lnum)]
# save results and dictionary for F construction on next steps
list_res = [[] for i in range(lnum-1)] # full results on each step. On first it is the F. in oh format
#dictm_prevstep = {} # save dictm for the next step
list_dictm = [[] for i in range(lnum-1)]# like resultst save dict on each step, for make F on the next step
resnooptimal = 0
#list_mvar[0] = data_ini['F'].to_list()
#list_res[0] = data_ini['F'].to_list()
#list_mvar.append(data_ini['F'].to_list())
# dict_varsres = {} # это не нужно ты знаешь номер каждой переменной в результирующем векторе, просто доставай из него

# Create matrises and ini of M_values
# for i in solvesteps_s:
#     list_M.append(m_create_byname(i[1].lower() + '_', dim))
#     #list_M_vals_m = np.zeros((dim*dim,dim))

for i in range(1,currmatrnum+1):
    list_M.append(m_create_byname('m'+str(i)+ '_', dim))
#print(list_M)
# if resnooptimal == 1:
#     break
# предполагаю, что мне приходит список списков. Нужна защита если передадут ['M1', 'M2l1l4', 'M3l2l3']
solvestep = solvesteps_s

dictm = {} # contain variables numbers
lolm = []
ldim_rst = []
dictm_cnt = 0   

# теперь заполнить словарь переменными матриц. Порядок от НА (стб-стр)
# fill in order
for mnum in range(currmatrnum):
    rows = list_M[mnum].shape[0]
    cols = list_M[mnum].shape[1]
    #print("rows, cols", rows, cols)
    for i in range(dim):
        for j in range(cols):
            for d in range(dim):
                if dictm.get(list_M[mnum][i*dim+d,j]) is None:
                    dictm.setdefault(list_M[mnum][i*dim+d,j], dictm_cnt)
                    dictm_cnt += 1
# fill ldim_rst for matrices ограничения по размерности
for mnum in range(currmatrnum):
    for j in range(list_M[mnum].shape[1]):
        for i in [i for i in range(list_M[mnum].shape[0]) if not i%dim]:
            tl = []
            for k in range(dim):
                #print(list_M[solvestep_num][i+k,j])
                tl.append(list_M[mnum][i+k,j]) 
            ldim_rst.append(tl)
#print('\n ldim_rst \n', ldim_rst)
    
# проверка конфл. должна производиться для переобозначенных переменных
# доступ к ним не через систему веток как прежде, а через список переобозначений (appel_list), 
# в нем указано какой переменной какие столбцы соответсвуют
# составление таблицы конфликтов

# Составление функционала
ressum = ''
dfconfl = pd.DataFrame(columns=[i for i in range(3)],index=range(len(data_step)))
dfconfl[2] = pd.Series(data_step["F"])
for i in range(exnum):
#for i in range(1):
    list_l = [np.nan for i in range(lnum+1)] # ini of list_l
    #print(list_l)
    lnames = get_leaves_txt(listToString(solvesteps_s))
    lnames.sort() # т.к. ф-я note_simplify_id выбирает из списка листов по 
                    # номеру - не надо их располагать в соотв-ии с записью
    #print('lnames in functional creation', lnames)
    # заполняем список листов значениями переменных из таблицы
    # заполняем таблицу конликтов значениями переменных из таблицы
    dfconfl_var1 = 'u_'
    for y in lnames:
        lnames_t = data_step.loc[i,y]
        #print('lnames dt content', int(lnames_t))
        #list_l.append(i2oh(int(lnames_t), dim)) # нужно размещать листы по их номерам, для работы со свернутой записью
        list_l[get_leaf_number(y)-1] = i2oh(int(lnames_t), dim)
        # нужно составить имя из числовых листов
        dfconfl_var1 +=  str(lnames_t)
    dfconfl.loc[i,0] = dfconfl_var1
    # плюс добавить переменную листов из свернутой части
    if len(appel_list):
        lnames_conv = data_step.loc[i,appel_list]
        lnames_conv = listToString(lnames_conv.values)
        lnames_conv = solvesteps_s[len(solvesteps_s)-1] + '_' + lnames_conv
        #list_l.append(makedimension(lnames_conv, dim))   
        list_l[len(list_l)-1] = makedimension(lnames_conv, dim)
        t_4_ldim_rst = [] # accumulate parts of variable
        #for j in makedimension(lnames_conv, dim):
        for j in list_l[len(list_l)-1]:
            if dictm.get(j) is None:
                dictm.setdefault(j, dictm_cnt)
                dictm_cnt += 1
                t_4_ldim_rst.append(j)
        if len(t_4_ldim_rst): 
            ldim_rst.append(t_4_ldim_rst) # dim restrictions
        dfconfl.loc[i,1] = lnames_conv
    #print('\n dfconfl \n', dfconfl)
    #print('\n ldim_rst \n', ldim_rst)
    
     
    
    
    note_cnt = 0
    note_t = []
    txt_slct = []
    list_l_t = list_l # it is not copy in python
    list_m_t = list_M
    note_t = copy.copy(solvesteps_s)
    #print(note_t)
    #print(list_l_t)
    #print(list_m_t)
    while(len(note_t) > 1):
        #print("note_t before",note_t)
        #print("list_l_t before", list_l_t)
        note_t, txt_slct, numOfFirstListInPair, numOfSecondListInPair\
        = note_simplify_id(note_t, list_l_t, list_m_t, dim)

        list_l_t[numOfFirstListInPair] = txt_slct            # need reini first in w pair
        list_l_t[numOfSecondListInPair] = np.nan        # in the pos of second used list write NA - it is out of scope
        list_m_t = list_m_t[0:(len(list_m_t)-1)]
        #print("note_t after", note_t)
        #print("list_l_t after", list_l_t)
        #print(list_m_t)
        #print("txt_slct", txt_slct)

    #print("txt_slct", txt_slct)
    #print("list_l", list_l)

    txt = txt_slct
    #print("txt", txt)
    # нужно упростить уравнение.
    # take F as F from tbl or as previous step result
#         if solvestep_num == 0: F = int(data_ini.loc[i,['K']])
#         else: F = list_mvar[solvestep_num]
    #F = list_mvar[solvestep_num][i]
    F = data_step['F'][i]
    #print('F', F)
    # нужно выбирать компоненту txt[х] в зависимости от значения функции, для данного примера
    res = polynomial_simplify(txt[F])
    #print('res ', i, res)
    ressum += "+" + res
#print('ressum', ressum)
#print("dictm", dictm)



# seek for conflicts in dfconfl table        
restrictions = []
investigatebothcols = {0}
#print("isinstance(dfconfl.iloc[0,0], int)", isinstance(dfconfl.iloc[0,0], int))
if isinstance(dfconfl.iloc[0,0], int): investigatebothcols = {0}  # only 1 and 0 in first(r) col
else: investigatebothcols = {0,1} # need to seek confl in both columns
# т.к. свернутая часть всегда в конце - первый столбец всегда числа. на них не м.б. конфликтов, иначе в данных были бы противоречия
investigatebothcols = {0}
for col in investigatebothcols: # смена ведущего столбца
    #print("col", col)
    col_diversity = set(dfconfl[col])
    for i in col_diversity:
        #print("--i--", i)
        lstore = []
        rstr = []
        dfconfl_s = dfconfl[dfconfl[col] == i]
        #print("dfconfl_s \n", dfconfl_s)
        if len(dfconfl_s) > 1 and len(set(dfconfl_s[2])) > 1: # there are confls
            for j in range(dim):
                #print("i, j", i, j)
                lstore_loc = []
                col_selected = {0,1}
                col_selected.discard(col)
                group = dfconfl_s[dfconfl_s[2] == j][col_selected]
                #print("  group \n", group)
                lstore_loc = list(chain.from_iterable(group.values))
                if len(lstore_loc): lstore.append(lstore_loc)
            #print(lstore)

            if len(lstore):
                rstr = makerestriction_pair(lstore)
                #print(rstr)
                for k in rstr:
                    restrictions.append(k)
#print('restrictions', restrictions)
# нужно удалить дублирующиеся ограничения

restrictions_sep = []
# не через умножение, а через перечисление как в Clauses
for i in restrictions:
    pparts = get_productparts(i)     
    print(pparts)
    txt = []
    t = []
    for i, j in zip(makedimension(pparts[0], dim), makedimension(pparts[1], dim)):
        t.append([i, j])
    txt.append(list(chain.from_iterable(t)))
    restrictions_sep.append(list(chain.from_iterable(txt)))
#print('\n restrictions_sep \n', restrictions_sep)

    


# составление групп мономов
Clauses_polymn = get_monomials(ressum)
#print("\n Clauses_polymn \n", Clauses_polymn)
# составить из них "and" clauses а потом сумму из этих clauses
Clauses_polymn_sep = []
for i in Clauses_polymn:
    pparts = get_productparts(i)
    Clauses_polymn_sep.append(pparts)
#print("\n Clauses_polymn_sep \n", Clauses_polymn_sep)
print("len(Clauses_polymn_sep)", len(Clauses_polymn_sep))

try:

    # Create a new model
    model = gp.Model("Genconstr")
    # initialize decision variables and objective
    Vars = model.addVars(len(dictm), vtype=GRB.BINARY, name="Vars")
    #Rst = model.addVars(len(restrictions_sep), vtype=GRB.BINARY, name="Rst")
    Cls = model.addVars(len(Clauses_polymn_sep), vtype=GRB.BINARY, name="Cls")
    #Obj0 = model.addVar(vtype=GRB.BINARY, name="Obj0")
    Obj0 = model.addVar( name="Obj0")
    #Obj1 = model.addVar(vtype=GRB.BINARY, name="Obj1")
    # создаем дополнительные переменные для непоследовательного случая, т.е. умножения 3-х членов
#     if len(Clauses_polymn_sep[0]) > 2:
#         Mul = model.addVars(len(Clauses_polymn_sep), vtype=GRB.BINARY, name="Mul")

    # добавь инициализацию в ограничения
    #model.addConstr((Vars[pow(dim,3)] == 1 ), name="CNSTR_varini"+str(0))

    # Link Xi and notXi from ldim_rst
    for i in range(len(ldim_rst)):
        #print('restr dim pairs', ldim_rst[i][0], ldim_rst[i][1])
        #model.addConstrs((ldim_rst[i][0] + ldim_rst[i][1] == 1.0 for i in range(NLITERALS)),
        #for j in range(len(ldim_rst[i])):
        objj = gp.quicksum(Vars[dictm[ldim_rst[i][j]]] for j in range(len(ldim_rst[i])))
        #print(objj)

        # переменные бинарные, м.б. ограничения не бинарные?
        model.addConstr((objj == 1 ), name="CNSTR_Dim"+str(i))

    # Make restrictions: Link restrictions_sep and literals
#     for i in range(len(restrictions_sep)):
#         #print('i, len(restrictions_sep)',i, len(restrictions_sep))
#         #print('restr pairs', restrictions_sep[i][0], restrictions_sep[i][1])
#         #print('restr ', restrictions_sep[i])
#         #for j in range(len(restrictions_sep[i])):
#         #    if not j%dim: print("restrictions_sep[i][j], restrictions_sep[i][j+1]", restrictions_sep[i][j], restrictions_sep[i][j+1])
#         objj = gp.quicksum(Vars[dictm[restrictions_sep[i][j]]] * Vars[dictm[restrictions_sep[i][j+1]]] 
#                           for j in range(len(restrictions_sep[i])) if not j%2) # pairs in monome
#         #######model.addConstr(Rst[i] == objj, name = 'CNSTR_Rst'+str(i))
#         model.addConstr((objj == 0), name = 'CNSTR_Rst'+str(i))

        # CNSTR_Rst д.б. бинарные. В сетапе это не указано

    # добавь инициализацию первой переменной в ограничениях 
    #Vars[dictm[restrictions_sep['m2_000_0'] = 1
    # Vars[dictm[restrictions_sep['m2_000_1'] = 0
    # измени порядок создания переменных - по столбцам - тогда сможешь просто инициализировать первые dim

    # Link objs with clauses
    #model.addConstr(Obj0 == gp.min_(Cla), name="CNSTR_Obj0")
    #model.addConstr((Obj1 == 1) >> (Cla.sum() >= 4.0), name="CNSTR_Obj1")

    # Make obj from polynomial: Link Clauses_polymn_sep and literals
    for i in range(len(Clauses_polymn_sep)):
        obj =  gp.and_(Vars[dictm[Clauses_polymn_sep[i][j]]] for j in range(len(Clauses_polymn_sep[i])) )
        model.addConstr(Cls[i] == obj, name = 'CNSTR_Cls'+str(i))

    #model.addConstr(Obj0 == gp.or_(Cls), name="CNSTR_Obj0")
    model.addConstr(Obj0 == gp.quicksum(Cls), name="CNSTR_Obj0")
    # Set optimization objective
    model.setObjective(Obj0, GRB.MAXIMIZE)

    # Save problem
    model.write("genconstr.mps")
    model.write("genconstr.lp")

    model.setParam('TimeLimit', 6)
    model.Params.PoolSearchMode = 2
    model.Params.PoolSolutions = 20

    # Optimize
    model.optimize()
    # Status checking
    status = model.getAttr(GRB.Attr.Status)

    if status in (GRB.INF_OR_UNBD, GRB.INFEASIBLE, GRB.UNBOUNDED):
        print("The model cannot be solved because it is infeasible or "
              "unbounded")
        sys.exit(1)
    if status != GRB.OPTIMAL:
        print("Optimization was stopped with status ", status)
        sys.exit(1)


    # Print result
    objval = model.getAttr(GRB.Attr.ObjVal)
    print("objval ", objval)
    if len(Clauses_polymn_sep[0]) > 2: # noconsequential structure
        print("pow(dim,len(Clauses_polymn_sep[0])-1)", pow(dim,len(Clauses_polymn_sep[0])-1))
        print(len(Clauses_polymn_sep)/pow(dim,len(Clauses_polymn_sep[0])-1))
        #if objval == (len(Clauses_polymn_sep)/dim/dim): 
        if objval == (len(Clauses_polymn_sep)/pow(dim,len(Clauses_polymn_sep[0])-1)): 
            print("\n found solution \n")
        else:
            resnooptimal = 1
            print("\n there is no exact solution for nonseq struct \n")
    else:
        if objval == len(Clauses_polymn_sep)/dim:
            print("\n found solution \n")
        else:
            resnooptimal = 1
            print("\n there is no exact solution \n")


except gp.GurobiError as e:
    print('Error code ' + str(e.errno) + ": " + str(e))

except AttributeError:
    print('Encountered an attribute error')

print('vars in polynome', len(dictm)-pow(dim,3))
print('monoms in polynome', len(Clauses_polymn_sep))
#print(len(restrictions_sep))
print(len(model.x))
print("model.x", model.x)
print("ohvec2i(model.x[0:len(dictm)], dim)", ohvec2i(model.x[0:len(dictm)], dim))

# not correct to use append if on input complex structure
#list_M_vals.append(ohvec2i(model.x[0:pow(dim,3)], dim))
for q in range(currmatrnum):
    list_M_vals.append(ohvec2i(model.x[q*pow(dim,3):(q+1)*pow(dim,3)], dim))
#list_M_vals_m.append(np.array(list_split(ohvec2i(model.x[0:pow(dim,3)], dim), dim)).T)
#list_mvar.append(ohvec2i(model.x[pow(dim,3):len(dictm)], dim)) # it should be placement in certain cell, it matrix order not [1,n]
#list_mvar[int(name_m[1:])] = ohvec2i(model.x[pow(dim,3):len(dictm)], dim)

# print('name_m', name_m)
# print(int(name_m[1:])) # step_f_num
# list_res[int(name_m[1:])] = model.x[0:len(dictm)]
# list_dictm[int(name_m[1:])]= dictm
#list_res.append(model.x[0:len(dictm)])
#dictm_prevstep = dictm
#list_dictm.append(dictm)
#tmvar = ohvec2i(model.x[pow(dim,3):len(dictm)], dim)
pool_size = model.getAttr(GRB.Attr.SolCount)
# нужно проверить все ли решения в пуле максимальны. Думаю, что эт не так и нужно будет их доп. проверять


print("my solcount", pool_size)
#print("list_mvar ", list_mvar)
#print("oh(list_res) ", ohvec2i(model.x[0:len(dictm)], dim))
#print("current list_m", ohvec2i(model.x[0:pow(dim,3)], dim))
print("current list_M_vals", np.array(list_split(ohvec2i(model.x[0:pow(dim,3)], dim), dim)).T)
a = 1
model.setParam(GRB.Param.SolutionNumber, a)
# print solutions
for number in range(1,pool_size):
    #if  number != a:
    model.setParam(GRB.Param.SolutionNumber, number)
    print(number, "ObjVal", model.getAttr(GRB.Attr.PoolObjVal), 
          (ohvec2i(model.xn[0:pow(dim,3)*(currmatrnum+1)], dim)), '\n')

     
        
        
xini = ohvec2i(model.x[0:pow(dim,3)], dim) # это размер одной матрицы
solutionssame = []
#solutionssame.append(xini)
#print('x0: \n', ohvec2i(model.x[0:len(dictm)], dim))
#print('xn: \n', ohvec2i(model.xn[0:len(dictm)], dim))

# select the same by matrix solutions
# for number in range(1,pool_size):
#     if  number != a:
#         model.setParam(GRB.Param.SolutionNumber, number)
#         if not any(np.array(ohvec2i(model.xn[0:pow(dim,3)], dim)) - np.array(xini)): # это размер одной матрицы
#             solutionssame.append(list(ohvec2i(model.xn[0:len(dictm)], dim)))
# print("len solutionssame", len(solutionssame))
# for i in solutionssame:
#      print(i, '\n')



print("final list_M_vals", list_M_vals)
#print("final list_mvar", list_mvar)
print("final list_M_vals_m", list_M_vals_m)
print("list_res", list_res)
print("list_dictm", list_dictm)

следи за правильностью заполнения хвоста
dictm {'m1_00_0': 0, 'm1_00_1': 1, 'm1_00_2': 2, 'm1_00_3': 3, 'm1_10_0': 4, 'm1_10_1': 5, 'm1_10_2': 6, 'm1_10_3': 7, 'm1_20_0': 8, 'm1_20_1': 9, 'm1_20_2': 10, 'm1_20_3': 11, 'm1_30_0': 12, 'm1_30_1': 13, 'm1_30_2': 14, 'm1_30_3': 15, 'm1_01_0': 16, 'm1_01_1': 17, 'm1_01_2': 18, 'm1_01_3': 19, 'm1_11_0': 20, 'm1_11_1': 21, 'm1_11_2': 22, 'm1_11_3': 23, 'm1_21_0': 24, 'm1_21_1': 25, 'm1_21_2': 26, 'm1_21_3': 27, 'm1_31_0': 28, 'm1_31_1': 29, 'm1_31_2': 30, 'm1_31_3': 31, 'm1_02_0': 32, 'm1_02_1': 33, 'm1_02_2': 34, 'm1_02_3': 35, 'm1_12_0': 36, 'm1_12_1': 37, 'm1_12_2': 38, 'm1_12_3': 39, 'm1_22_0': 40, 'm1_22_1': 41, 'm1_22_2': 42, 'm1_22_3': 43, 'm1_32_0': 44, 'm1_32_1': 45, 'm1_32_2': 46, 'm1_32_3': 47, 'm1_03_0': 48, 'm1_03_1': 49, 'm1_03_2': 50, 'm1_03_3': 51, 'm1_13_0': 52, 'm1_13_1': 53, 'm1_13_2': 54, 'm1_13_3': 55, 'm1_23_0': 56, 'm1_23_1': 57, 'm1_23_2': 58, 'm1_23_3': 59, 'm1_33_0': 60, 'm1_33_1': 61, 'm1_33_2': 62, 'm1_33_3': 63, 'm

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



Set parameter TimeLimit to value 6
Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 20
Gurobi Optimizer version 9.5.0 build v9.5.0rc5 (win64)
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads
Optimize a model with 145 rows, 3736129 columns and 3736129 nonzeros
Model fingerprint: 0xf7238d9c
Model has 3735552 general constraints
Variable types: 1 continuous, 3736128 integer (3736128 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 0 rows and 0 columns (presolve time = 15s) ...
Presolve removed 0 rows and 0 columns (presolve time = 15s) ...
Presolve removed 0 rows and 0 columns (presolve time = 23s) ...
Presolve added 37355520 rows and 0 columns
Presolve time: 24.11s

Explored 0 nodes (0 simplex iterations) in 25.24 seconds (10.63 work units)
Thread count was 1 (of 16 available processors)

Solution 

SystemExit: 1

C:\ProgramData\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3449: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [41]:
tb = [[1, 2, 0], [1, 2, 3]]
t1 = [1, 2, -0]
if t1 in tb: print("get")

get


In [91]:
df1 = pd.DataFrame(list_M_vals_m[0])
for i in range(1,len(list_M_vals_m)):
    #print(df1)
    df2 = pd.DataFrame(list_M_vals_m[i])
    df1 = pd.concat([df1, df2], axis=0, join='outer')
df1.to_excel('212114_arct_l5d3_s13_matrices.xlsx', engine="xlsxwriter", index=False)

